# Face Recognition Program

In [78]:
import cv2
import os
import face_recognition as fcr
import numpy as np
from typing import List
from typing import Tuple

In [79]:
class ImageList :
    def __init__(self, list_names: List[str], base_path) -> None:
        # constructor
        self.image_map = {}
        
        self.known_encoded = []
        self.known_names = []
        self.base_path = base_path
        self.total_images = 0

        print('Retrieving images', end='.')
        for name in list_names : 
            print('.',end='.')
            self.image_map[name] =  self.get_all_img(name)
            self.total_images += len(self.image_map[name])
        print('\nRetrieval done')    
        
    def get_base_path(self) -> str :
        return self.base_path

    def get_all_img(self, person_name: str) -> List[str] :
        file_names = os.listdir(f'{self.get_base_path()}{person_name}/')
        return file_names
    
    def encode_all(self) -> None :
        print(f'Encoding {self.total_images} images')
        for key in self.image_map.keys() :
            print(f'Encoding {key} images in process')
            for i in range(len(self.image_map[key])) :
                img = cv2.imread(self.fetch_one(key, i))

                # need to convert color since opencv use bgr and face_recognition use rgb
                rgb_img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                face_encodings = fcr.face_encodings(rgb_img)
                
                if len(face_encodings) == 0 :
                    print(f'No face detected in image file: {self.fetch_one(key, i)}')
                    continue
                
                img_encoded = face_encodings[0]

                self.known_encoded.append(img_encoded)
                self.known_names.append(key)
            print(f'Encoding {key} images done')

    def fetch_one(self, name: str, idx: int) -> str :
        return f'{self.base_path}{name}/{self.image_map[name][idx]}'
    
    def detect_faces(self, frame) -> Tuple[int, List[str]] :
        small_frame = cv2.resize(frame, (0, 0), fx=0.25, fy=0.25)
        
        # find all face and face encodings in the current frame
        rgb_small = cv2.cvtColor(small_frame, cv2.COLOR_BGR2RGB)
        face_locations = fcr.face_locations(rgb_small)
        face_encodings = fcr.face_encodings(rgb_small, face_locations)

        face_names = []
        for face_encoding in face_encodings :
            matches = fcr.compare_faces(self.known_encoded, face_encoding)
            name = 'ndak tau saya'

            # if match was found, use the first one
            if True in matches :
                match_index = matches.index(True)
                name = self.known_names[match_index]

            face_distances = fcr.face_distance(self.known_encoded, face_encoding)
            best_match_index = np.argmin(face_distances)
            if matches[best_match_index] :
                name = self.known_names[match_index]

            face_names.append(name)

        face_locations = np.array(face_locations)
        face_locations = face_locations / 0.25
        return face_locations.astype(int), face_names

In [80]:
def play(img_list: ImageList) :
    if (len(img_list.known_encoded) == 0) :
        img_list.encode_all()
    
    print('Opening webcam')
    cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)

    window_name = 'Camera'

    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 800)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 600)

    while True :
        ret, frame = cap.read()

        if not ret :
            print('Error: could not read frame')
            break 

        # trying to detect face
        face_locs, names = img_list.detect_faces(frame)

        for face_loc, name in zip(face_locs, names) :
            y1, x2, y2, x1 = face_loc[0], face_loc[1], face_loc[2], face_loc[3]

            cv2.putText(frame, name, (x1, y1 - 10), cv2.FONT_HERSHEY_DUPLEX, 1, (0,0,0), 2)
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 200), 4)

        cv2.imshow(window_name, frame)
        key = cv2.waitKey(1)

        if key == 27 or key == 113 :
            break 

    cap.release()
    cv2.destroyAllWindows()

In [81]:
play(img_list=ImageList(['dayu', 'devan', 'leo'], 'img/'))

Retrieving images.......Retrieval done
Encoding 27 images
Encoding dayu images in process
Encoding dayu images done
Encoding devan images in process
No face detected in image file: img/devan/IMG_20231212_182010.jpg
No face detected in image file: img/devan/IMG_20231212_182058.jpg
Encoding devan images done
Encoding leo images in process
Encoding leo images done
